## Outline

- DF for minute features at date grain (Done)
- DF for daily features at date grain (Done)
- DF for returns over 1, 3 and 5 days (Done)
- Simple logistic, rfc and xgb models for daily alone, min alone and then combined
- Permutation importance
- Chart over rolling 5 days for 25 iterations, aka 6 months

In [1]:
import min_features, daily_return
import importlib
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

importlib.reload(min_features)
importlib.reload(daily_return)

df_min = min_features.min_features()
returns = [1, 3, 5, 10]
df_daily = daily_return.pull_daily('QQQ', returns) 

df_main = pd.merge(df_min, df_daily, how='inner', on='Date')
df_main = df_main.sort_values(by='Date', ascending=False)

return_cols = df_main.columns[df_main.columns.str.contains("Return_")].to_list()
daily_cols = [
    c for c in df_daily.iloc[:, 1:].columns
    if "return" not in c.lower()
]
min_cols = df_min.iloc[:, 1:].columns.to_list()

/Users/brettchase/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)

# -----------------------------
# Models
# -----------------------------
models = {
    "xgboost": XGBClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    "random_forest": RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
}

# -----------------------------
# Helpers
# -----------------------------
def _compute_dist(y):
    """Distribution stats for y in {0,1}."""
    n = int(len(y))
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    return {
        "test_n": n,
        "test_pos_n": n_pos,
        "test_neg_n": n_neg,
        "test_pos_frac": (n_pos / n) if n else np.nan,
        "test_neg_frac": (n_neg / n) if n else np.nan,
    }

def walkback_runs(
    df,
    feature_cols,
    target_col,
    *,
    date_col="Date",
    train_years=6,
    test_days=5,
    step_days=5,
    runs=20,
    horizon_days=1,        # r (used for purge)
    purge_days=None,       # defaults to horizon_days
    fill_inf=0.0,
):
    """
    Deployment-aligned evaluation:
      - For each run, take a 5-day OOT test window stepping back by 5 days.
      - Train on the prior N years (fixed-length window) ending right before test.
      - Purge 'purge_days' from the end of train to avoid overlap leakage for forward-return labels.
      - Score ONLY on the OOT test window (distribution + metrics).
    Returns: long DataFrame with one row per (feature_set/run/model).
    """
    dfw = df.sort_values("Date").reset_index(drop=True).copy()

    # Drop any accidental return cols from features (belt+suspenders)
    safe_feature_cols = [c for c in feature_cols if "Return" not in c]

    # Basic numeric cleaning
    dfw[safe_feature_cols] = dfw[safe_feature_cols].replace([np.inf, -np.inf], fill_inf)

    n = len(dfw)
    train_size = 245 * int(train_years)
    test_size = int(test_days)
    step = int(step_days)
    purge = int(purge_days) if purge_days is not None else 0 #int(horizon_days)

    X_all = dfw[safe_feature_cols].to_numpy()
    #y_all = _to_binary(dfw[target_col].to_numpy())
    y_all = dfw[target_col].to_numpy()
    dates = dfw[date_col].to_numpy() if date_col in dfw.columns else None

    rows = []

    for k in range(runs):
        test_end = n - k * step
        test_start = test_end - test_size
        if test_start < 0:
            break

        train_end = test_start - purge
        train_start = train_end - train_size
        if train_start < 0 or train_end <= train_start:
            break

        print(
            f"Run {k+1}/{runs} | "
            f"Train: {dates[train_start]} → {dates[train_end-1]} | "
            f"Test: {dates[test_start]} → {dates[test_end-1]} | "
            f"Train_n={train_end-train_start} | Test_n={test_end-test_start}"
        )

        X_train = X_all[train_start:train_end]
        y_train = y_all[train_start:train_end]
        X_test  = X_all[test_start:test_end]
        y_test  = y_all[test_start:test_end]

        dist = _compute_dist(y_test)
        single_class_test = (np.unique(y_test).size < 2)

        for model_name, model in models.items():
            m = clone(model)
            m.fit(X_train, y_train)

            preds = m.predict(X_test)

            # probabilities if available (for confidence metrics)
            proba = None
            if hasattr(m, "predict_proba"):
                proba = m.predict_proba(X_test)[:, 1]
            elif hasattr(m, "decision_function"):
                s = m.decision_function(X_test)
                # squash to (0,1) so confidence metrics work consistently
                proba = 1.0 / (1.0 + np.exp(-s))

            # confidence/coverage metrics (optional but useful)
            topk_acc = np.nan
            topk_cov = np.nan
            if proba is not None and len(proba) > 0:
                conf = np.abs(proba - 0.5)
                # top 40% by confidence (with 5 samples, this is ~2 samples)
                q = np.quantile(conf, 0.60)
                sel = conf >= q
                topk_cov = float(sel.mean())
                topk_acc = float((preds[sel] == y_test[sel]).mean()) if sel.any() else np.nan

            rows.append({
                "run": k + 1,
                "model": model_name,

                # core metrics
                "bal_acc": float(balanced_accuracy_score(y_test, preds)),
                "acc": float(accuracy_score(y_test, preds)),
                "sign_acc": 2 * float(accuracy_score(y_test, preds)) - 1,
                "mcc": float(matthews_corrcoef(y_test, preds)),

                # only meaningful if test has both classes
                "f1": np.nan if single_class_test else float(f1_score(y_test, preds, zero_division=0)),
                "precision": np.nan if single_class_test else float(precision_score(y_test, preds, zero_division=0)),
                "recall": np.nan if single_class_test else float(recall_score(y_test, preds, zero_division=0)),

                # confidence-conditioned performance (if proba/decision_function exists)
                "top40_acc": topk_acc,
                "top40_cov": topk_cov,

                **dist,

                "train_n": int(len(y_train)),
                "train_start": dates[train_start] if dates is not None else train_start,
                "train_end": dates[train_end - 1] if dates is not None else train_end - 1,
                "test_start": dates[test_start] if dates is not None else test_start,
                "test_end": dates[test_end - 1] if dates is not None else test_end - 1,
                "train_years": train_years,
                "horizon_days": horizon_days,
                "n_features": len(safe_feature_cols),
            })

    return pd.DataFrame(rows)

# -----------------------------
# Run grid (feature sets x horizon x train_years, etc.)
# -----------------------------
column_sets = [daily_cols, min_cols, daily_cols + min_cols]
names = ["daily", "minute", "daily+minute"]

returns = [1]#, 3, 5, 10]#, 3, 5]  # add 3,5,etc later
train_years_grid = [5]#[3, 5, 7]  # could be [3,4,5,6]
runs = 1
test_days = 5
step_days = 5

results_all = []
results_np = []
results_no_purge = pd.DataFrame()

for feature_cols, feat_name in zip(column_sets, names):
    for r in returns:
        print(f"{r} | {feat_name}")
        target_col = f"Return_{r}"

        for train_years in train_years_grid:
            df_scores = walkback_runs(
                df=df_main,
                feature_cols=feature_cols,
                target_col=target_col,
                date_col="Date",
                train_years=train_years,
                test_days=test_days,
                step_days=step_days,
                runs=runs,
                horizon_days=r,
                purge_days=None,   # purge = horizon (safe default)
                fill_inf=0.0,
            )

            df_scores["feature_set"] = feat_name
            df_scores["horizon"] = r

            results_np.append(df_scores)

results_baseline = pd.concat(results_np, ignore_index=True)

1 | daily
Run 1/1 | Train: 2021-01-15 → 2025-12-12 | Test: 2025-12-15 → 2025-12-19 | Train_n=1225 | Test_n=5
1 | minute
Run 1/1 | Train: 2021-01-15 → 2025-12-12 | Test: 2025-12-15 → 2025-12-19 | Train_n=1225 | Test_n=5
1 | daily+minute
Run 1/1 | Train: 2021-01-15 → 2025-12-12 | Test: 2025-12-15 → 2025-12-19 | Train_n=1225 | Test_n=5


In [3]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)

# -----------------------------
# Models
# -----------------------------
models = {
    "xgboost": XGBClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    "random_forest": RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
}

# -----------------------------
# Helpers
# -----------------------------
def _compute_dist(y):
    """Distribution stats for y in {0,1}."""
    n = int(len(y))
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    return {
        "test_n": n,
        "test_pos_n": n_pos,
        "test_neg_n": n_neg,
        "test_pos_frac": (n_pos / n) if n else np.nan,
        "test_neg_frac": (n_neg / n) if n else np.nan,
    }

def walkback_runs(
    df,
    feature_cols,
    target_col,
    *,
    date_col="Date",
    train_years=6,
    test_days=5,
    step_days=5,
    runs=20,
    horizon_days=1,        # r (used for purge)
    purge_days=None,       # defaults to horizon_days
    fill_inf=0.0,
):
    """
    Deployment-aligned evaluation:
      - For each run, take a 5-day OOT test window stepping back by 5 days.
      - Train on the prior N years (fixed-length window) ending right before test.
      - Purge 'purge_days' from the end of train to avoid overlap leakage for forward-return labels.
      - Score ONLY on the OOT test window (distribution + metrics).
    Returns: long DataFrame with one row per (feature_set/run/model).
    """
    dfw = df.sort_values("Date").reset_index(drop=True).copy()

    # Drop any accidental return cols from features (belt+suspenders)
    safe_feature_cols = [c for c in feature_cols if "Return" not in c]

    # Basic numeric cleaning
    dfw[safe_feature_cols] = dfw[safe_feature_cols].replace([np.inf, -np.inf], fill_inf)

    n = len(dfw)
    train_size = 245 * int(train_years)
    test_size = int(test_days)
    step = int(step_days)
    purge = int(purge_days) if purge_days is not None else 0 #int(horizon_days)

    X_all = dfw[safe_feature_cols].to_numpy()
    #y_all = _to_binary(dfw[target_col].to_numpy())
    y_all = dfw[target_col].to_numpy()
    dates = dfw[date_col].to_numpy() if date_col in dfw.columns else None

    rows = []

    for k in range(runs):
        test_end = n - k * step
        test_start = test_end - test_size
        if test_start < 0:
            break

        train_end = test_start - purge
        train_start = train_end - train_size
        if train_start < 0 or train_end <= train_start:
            break

        print(
            f"Run {k+1}/{runs} | "
            f"Train: {dates[train_start]} → {dates[train_end-1]} | "
            f"Test: {dates[test_start]} → {dates[test_end-1]} | "
            f"Train_n={train_end-train_start} | Test_n={test_end-test_start}"
        )

        X_train = X_all[train_start:train_end]
        y_train = y_all[train_start:train_end]
        X_test  = X_all[test_start:test_end]
        y_test  = y_all[test_start:test_end]

        dist = _compute_dist(y_test)
        single_class_test = (np.unique(y_test).size < 2)

        for model_name, model in models.items():
            m = clone(model)
            m.fit(X_train, y_train)

            preds = m.predict(X_test)

            # probabilities if available (for confidence metrics)
            proba = None
            if hasattr(m, "predict_proba"):
                proba = m.predict_proba(X_test)[:, 1]
            elif hasattr(m, "decision_function"):
                s = m.decision_function(X_test)
                # squash to (0,1) so confidence metrics work consistently
                proba = 1.0 / (1.0 + np.exp(-s))

            # confidence/coverage metrics (optional but useful)
            topk_acc = np.nan
            topk_cov = np.nan
            if proba is not None and len(proba) > 0:
                conf = np.abs(proba - 0.5)
                # top 40% by confidence (with 5 samples, this is ~2 samples)
                q = np.quantile(conf, 0.60)
                sel = conf >= q
                topk_cov = float(sel.mean())
                topk_acc = float((preds[sel] == y_test[sel]).mean()) if sel.any() else np.nan

            rows.append({
                "run": k + 1,
                "model": model_name,

                # core metrics
                "bal_acc": float(balanced_accuracy_score(y_test, preds)),
                "acc": float(accuracy_score(y_test, preds)),
                "sign_acc": 2 * float(accuracy_score(y_test, preds)) - 1,
                "mcc": float(matthews_corrcoef(y_test, preds)),

                # only meaningful if test has both classes
                "f1": np.nan if single_class_test else float(f1_score(y_test, preds, zero_division=0)),
                "precision": np.nan if single_class_test else float(precision_score(y_test, preds, zero_division=0)),
                "recall": np.nan if single_class_test else float(recall_score(y_test, preds, zero_division=0)),

                # confidence-conditioned performance (if proba/decision_function exists)
                "top40_acc": topk_acc,
                "top40_cov": topk_cov,

                **dist,

                "train_n": int(len(y_train)),
                "train_start": dates[train_start] if dates is not None else train_start,
                "train_end": dates[train_end - 1] if dates is not None else train_end - 1,
                "test_start": dates[test_start] if dates is not None else test_start,
                "test_end": dates[test_end - 1] if dates is not None else test_end - 1,
                "train_years": train_years,
                "horizon_days": horizon_days,
                "n_features": len(safe_feature_cols),
            })

    return pd.DataFrame(rows)

# -----------------------------
# Run grid (feature sets x horizon x train_years, etc.)
# -----------------------------
column_sets = [daily_cols, min_cols, daily_cols + min_cols]
names = ["daily", "minute", "daily+minute"]

top_feats = ['num_days_100', 'Vol_Ratio_10', 'VROC_3', 'VROC_10',
       'Vol_Spike_40', 'CMF_20', 'VIX', 'VROC_5', 'VIX_1_change',
       'Vol_Spike_10', 'Vol_Ratio_25', 'num_days_50', 'num_days_200',
       'OBV_Z5', 'Vol_Ratio_50', 'Max_120_Rows_Since', 'CMF_10', 'High',
       'SMA_100', 'ADL', 'VIX_10_change', 'Min_30_Rows_Since',
       'Close_Rel_Min200', 'Min_240_Rows_Since', 'SMA_200', 'Close',
       'max_min_first-30m', 'early_post_market_oc_pos_max',
       'late_pre_market_%_up_minutes', 'overnight_oc_neg_avg',
       'early_close_oc_neg_avg', 'midday_%_none_minutes',
       'post_market_other_oc_neg_avg', 'late_open_oc_pos_avg',
       'early_pre_market_oc_pos_max', 'midday_%_up_minutes',
       'overnight_oc_pos_avg', 'post_market_other_oc_pos_avg']


column_sets = [top_feats]
names = ["top_feats"]
returns = [1]#, 3, 5, 10]#, 3, 5]  # add 3,5,etc later
train_years_grid = [5]#[3, 5, 7]  # could be [3,4,5,6]
runs = 1
test_days = 5
step_days = 5

results= []
results_trimmed = pd.DataFrame()

for feature_cols, feat_name in zip(column_sets, names):
    for r in returns:
        print(f"{r} | {feat_name}")
        target_col = f"Return_{r}"

        for train_years in train_years_grid:
            df_scores = walkback_runs(
                df=df_main,
                feature_cols=feature_cols,
                target_col=target_col,
                date_col="Date",
                train_years=train_years,
                test_days=test_days,
                step_days=step_days,
                runs=runs,
                horizon_days=r,
                purge_days=None,   # purge = horizon (safe default)
                fill_inf=0.0,
            )

            df_scores["feature_set"] = feat_name
            df_scores["horizon"] = r

            results.append(df_scores)

results_trimmed = pd.concat(results, ignore_index=True)

1 | top_feats
Run 1/1 | Train: 2021-01-15 → 2025-12-12 | Test: 2025-12-15 → 2025-12-19 | Train_n=1225 | Test_n=5


In [27]:
results_baseline.columns

Index(['run', 'model', 'bal_acc', 'acc', 'sign_acc', 'mcc', 'f1', 'precision',
       'recall', 'top40_acc', 'top40_cov', 'test_n', 'test_pos_n',
       'test_neg_n', 'test_pos_frac', 'test_neg_frac', 'train_n',
       'train_start', 'train_end', 'test_start', 'test_end', 'train_years',
       'horizon_days', 'n_features', 'feature_set', 'horizon'],
      dtype='object')

In [4]:
results_baseline = pd.read_csv("baseline_performance_1-3-5-10.csv")

In [42]:
results_baseline.to_csv("baseline_performance_1-3-5-10.csv", index=False)

In [29]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
idx  = ["feature_set", "horizon", "train_years", "model"]

dfs = [results_baseline, results_trimmed]
final_df = None
metric = 'acc' #'signed_acc, acc

for df in dfs:
    df = df.copy()

    # bucket to exact 5-day bins
    df["test_pos_frac"] = ((df["test_pos_frac"] * 5).round() / 5).clip(0, 1)

    # overall (distribution-agnostic)
    overall = (
        df.groupby(idx, as_index=False)
          .agg(m_all=(metric, "mean"), n_all=("run", "count"))
    )

    # by-bin
    g = (
        df.groupby(idx + ["test_pos_frac"], as_index=False)
          .agg(m=(metric, "mean"), n=("run", "count"))
    )

    wide = (
        g.pivot(index=idx, columns="test_pos_frac", values=["m", "n"])
         .reindex(columns=bins, level=1)
    )
    wide.columns = [f"{metric}_{frac:g}" for metric, frac in wide.columns]
    wide = wide.reset_index()

    # merge overall into wide
    wide = wide.merge(overall, on=idx, how="left")

    # optional: column order
    column_order = ['feature_set', 'horizon', 'train_years', 'model', 'm_all', 'n_all', 'm_0', 'n_0', 'm_0.2', 
                    'n_0.2', 'm_0.4', 'n_0.4', 'm_0.6', 'n_0.6', 'm_0.8', 'n_0.8', 'm_1', 'n_1']

    wide = wide[column_order].round(3)

    if final_df is None:
        final_df = wide.copy()
    else:
        final_df = pd.concat([final_df, wide.copy()], ignore_index=True)

final_df.sort_values(by=['horizon', 'm_all'], ascending=False)

,feature_set,horizon,train_years,model,m_all,n_all,m_0,n_0,m_0.2,n_0.2,m_0.4,n_0.4,m_0.6,n_0.6,m_0.8,n_0.8,m_1,n_1
6,daily,10,5,random_forest,0.707,110,0.647,17.0,0.733,9.0,0.582,11.0,0.600,10.0,0.660,10.0,0.777,53.0
30,top_feats,10,5,random_forest,0.704,110,0.612,17.0,0.667,9.0,0.564,11.0,0.660,10.0,0.720,10.0,0.774,53.0
14,daily+minute,10,5,random_forest,0.684,110,0.494,17.0,0.511,9.0,0.564,11.0,0.560,10.0,0.700,10.0,0.819,53.0
31,top_feats,10,5,xgboost,0.664,110,0.424,17.0,0.622,9.0,0.436,11.0,0.620,10.0,0.660,10.0,0.804,53.0
22,minute,10,5,random_forest,0.656,110,0.047,17.0,0.200,9.0,0.418,11.0,0.620,10.0,0.800,10.0,0.958,53.0
15,daily+minute,10,5,xgboost,0.653,110,0.400,17.0,0.511,9.0,0.509,11.0,0.600,10.0,0.720,10.0,0.785,53.0
7,daily,10,5,xgboost,0.629,110,0.388,17.0,0.511,9.0,0.455,11.0,0.640,10.0,0.720,10.0,0.743,53.0
23,minute,10,5,xgboost,0.578,110,0.200,17.0,0.222,9.0,0.400,11.0,0.600,10.0,0.640,10.0,0.781,53.0
20,minute,5,5,random_forest,0.598,110,0.077,13.0,0.169,13.0,0.424,17.0,0.600,11.0,0.693,15.0,0.937,41.0
4,daily,5,5,random_forest,0.587,110,0.308,13.0,0.523,13.0,0.565,17.0,0.618,11.0,0.707,15.0,0.654,41.0


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
)
from sklearn.inspection import permutation_importance


def walkback_runs(
    df,
    feature_cols,
    target_col,
    *,
    models,                     # pass in models dict explicitly
    date_col="Date",
    train_years=6,
    test_days=5,
    step_days=5,
    runs=20,
    horizon_days=1,             # kept for metadata
    purge_days=None,            # default: 0 (prod-aligned)
    fill_inf=0.0,
    pi_tail_n=700,              # <-- PI computed on last N training records
    pi_scoring="balanced_accuracy",
    pi_repeats=5,
    pi_random_state=42,
    pi_n_jobs=-1,
):
    """
    Deployment-aligned evaluation:
      - For each run, take an OOT test window stepping back by step_days.
      - Train on prior N years ending right before test (optionally purged).
      - Score ONLY on OOT test.
      - Compute permutation importance on the *tail of training* (last pi_tail_n records).
    Returns:
      scores_df: one row per (run, model)
      pi_df: long PI table with one row per (run, model, feature)
    """
    dfw = df.sort_values(date_col).reset_index(drop=True).copy()

    # Drop any accidental return cols from features (belt+suspenders)
    safe_feature_cols = [c for c in feature_cols if "Return" not in c]

    # Basic numeric cleaning
    dfw[safe_feature_cols] = dfw[safe_feature_cols].replace([np.inf, -np.inf], fill_inf)

    n = len(dfw)
    train_size = 245 * int(train_years)
    test_size = int(test_days)
    step = int(step_days)
    purge = int(purge_days) if purge_days is not None else 0

    X_all = dfw[safe_feature_cols].to_numpy()
    y_all = dfw[target_col].to_numpy()
    dates = dfw[date_col].to_numpy()

    score_rows = []
    pi_rows = []

    for k in range(runs):
        test_end = n - k * step
        test_start = test_end - test_size
        if test_start < 0:
            break

        train_end = test_start - purge
        train_start = train_end - train_size
        if train_start < 0 or train_end <= train_start:
            break

        print(
            f"Run {k+1}/{runs} | "
            f"Train: {dates[train_start]} → {dates[train_end-1]} | "
            f"Test: {dates[test_start]} → {dates[test_end-1]} | "
            f"Train_n={train_end-train_start} | Test_n={test_end-test_start}"
        )

        X_train = X_all[train_start:train_end]
        y_train = y_all[train_start:train_end]
        X_test = X_all[test_start:test_end]
        y_test = y_all[test_start:test_end]

        single_class_test = (np.unique(y_test).size < 2)

        # PI window = last pi_tail_n of training
        tail_n = int(min(pi_tail_n, len(y_train)))
        X_pi = X_train[-tail_n:]
        y_pi = y_train[-tail_n:]

        for model_name, model in models.items():
            m = clone(model)
            m.fit(X_train, y_train)

            preds = m.predict(X_test)

            # optional: "top40" based on model output (not used for PI)
            proba = None
            if hasattr(m, "predict_proba"):
                proba = m.predict_proba(X_test)[:, 1]
            elif hasattr(m, "decision_function"):
                s = m.decision_function(X_test)
                proba = 1.0 / (1.0 + np.exp(-s))

            topk_acc = np.nan
            topk_cov = np.nan
            if proba is not None and len(proba) > 0:
                conf = np.abs(proba - 0.5)
                q = np.quantile(conf, 0.60)  # top 40%
                sel = conf >= q
                topk_cov = float(sel.mean())
                topk_acc = float((preds[sel] == y_test[sel]).mean()) if sel.any() else np.nan

            # --- permutation importance on TRAIN tail ---
            pi_mean = np.full(len(safe_feature_cols), np.nan, dtype=float)
            pi_std = np.full(len(safe_feature_cols), np.nan, dtype=float)

            # PI requires at least some label variability; if single-class, it can be meaningless
            if np.unique(y_pi).size >= 2 and tail_n >= 25:
                pi = permutation_importance(
                    m,
                    X_pi,
                    y_pi,
                    scoring=pi_scoring,
                    n_repeats=pi_repeats,
                    random_state=pi_random_state,
                    n_jobs=pi_n_jobs,
                )
                pi_mean = pi.importances_mean
                pi_std = pi.importances_std

                pi_rows.append(
                    pd.DataFrame(
                        {
                            "run": k + 1,
                            "model": model_name,
                            "feature": safe_feature_cols,
                            "pi_mean": pi_mean,
                            "pi_std": pi_std,
                            "pi_tail_n": tail_n,
                            "train_years": train_years,
                            "horizon_days": horizon_days,
                            "train_start": dates[train_start],
                            "train_end": dates[train_end - 1],
                            "test_start": dates[test_start],
                            "test_end": dates[test_end - 1],
                        }
                    )
                )

            score_rows.append(
                {
                    "run": k + 1,
                    "model": model_name,
                    "bal_acc": float(balanced_accuracy_score(y_test, preds)),
                    "acc": float(accuracy_score(y_test, preds)),
                    "sign_acc": 2.0 * float(accuracy_score(y_test, preds)) - 1.0,
                    "mcc": float(matthews_corrcoef(y_test, preds)),
                    "f1": np.nan if single_class_test else float(f1_score(y_test, preds, zero_division=0)),
                    "precision": np.nan if single_class_test else float(precision_score(y_test, preds, zero_division=0)),
                    "recall": np.nan if single_class_test else float(recall_score(y_test, preds, zero_division=0)),
                    "top40_acc": topk_acc,
                    "top40_cov": topk_cov,
                    "train_n": int(len(y_train)),
                    "test_n": int(len(y_test)),
                    "train_start": dates[train_start],
                    "train_end": dates[train_end - 1],
                    "test_start": dates[test_start],
                    "test_end": dates[test_end - 1],
                    "train_years": train_years,
                    "horizon_days": horizon_days,
                    "n_features": len(safe_feature_cols),
                    "pi_tail_n": tail_n,
                }
            )

    scores_df = pd.DataFrame(score_rows)
    pi_df = pd.concat(pi_rows, ignore_index=True) if len(pi_rows) else pd.DataFrame(
        columns=["run", "model", "feature", "pi_mean", "pi_std", "pi_tail_n",
                 "train_years", "horizon_days", "train_start", "train_end", "test_start", "test_end"]
    )
    return scores_df, pi_df


# -----------------------------
# Run grid (feature sets x horizon x train_years, etc.)
# -----------------------------
column_sets = [daily_cols, min_cols, daily_cols + min_cols]
names = ["daily", "minute", "daily+minute"]

returns = [1, 3, 5, 10]
train_years_grid = [5]
runs = 5
test_days = 5
step_days = 50

scores_all = []
pi_all = []

for feature_cols, feat_name in zip(column_sets, names):
    for r in returns:
        target_col = f"Return_{r}"
        print(f"{r} | {feat_name}")

        for train_years in train_years_grid:
            df_scores, df_pi = walkback_runs(
                df=df_main,
                feature_cols=feature_cols,
                target_col=target_col,
                models=models,
                date_col="Date",
                train_years=train_years,
                test_days=test_days,
                step_days=step_days,
                runs=runs,
                horizon_days=r,
                purge_days=None,   # prod-aligned
                fill_inf=0.0,
                pi_tail_n=700,
                pi_scoring="balanced_accuracy",
                pi_repeats=10,
                pi_random_state=42,
                pi_n_jobs=-1,
            )

            df_scores["feature_set"] = feat_name
            df_scores["horizon"] = r

            df_pi["feature_set"] = feat_name
            df_pi["horizon"] = r

            scores_all.append(df_scores)
            pi_all.append(df_pi)

results_pi_df = pd.concat(scores_all, ignore_index=True)
pi_long_df = pd.concat(pi_all, ignore_index=True)


1 | daily
Run 1/5 | Train: 2021-01-15 → 2025-12-12 | Test: 2025-12-15 → 2025-12-19 | Train_n=1225 | Test_n=5
Run 2/5 | Train: 2020-10-30 → 2025-10-01 | Test: 2025-10-02 → 2025-10-08 | Train_n=1225 | Test_n=5
Run 3/5 | Train: 2020-08-20 → 2025-07-22 | Test: 2025-07-23 → 2025-07-29 | Train_n=1225 | Test_n=5
Run 4/5 | Train: 2020-06-10 → 2025-05-07 | Test: 2025-05-08 → 2025-05-14 | Train_n=1225 | Test_n=5
Run 5/5 | Train: 2020-03-30 → 2025-02-25 | Test: 2025-02-26 → 2025-03-04 | Train_n=1225 | Test_n=5
3 | daily
Run 1/5 | Train: 2021-01-15 → 2025-12-12 | Test: 2025-12-15 → 2025-12-19 | Train_n=1225 | Test_n=5
Run 2/5 | Train: 2020-10-30 → 2025-10-01 | Test: 2025-10-02 → 2025-10-08 | Train_n=1225 | Test_n=5
Run 3/5 | Train: 2020-08-20 → 2025-07-22 | Test: 2025-07-23 → 2025-07-29 | Train_n=1225 | Test_n=5
Run 4/5 | Train: 2020-06-10 → 2025-05-07 | Test: 2025-05-08 → 2025-05-14 | Train_n=1225 | Test_n=5
Run 5/5 | Train: 2020-03-30 → 2025-02-25 | Test: 2025-02-26 → 2025-03-04 | Train_n=1225 |

In [ ]:
print(len(pi_long_df))
print(len(pi _long_df[pi_long_df['pi_mean'] > 0]))

14480
183


In [19]:
(pi_long_df['feature'][pi_long_df['pi_mean'] > 0].unique())

array(['num_days_100', 'Vol_Ratio_10', 'VROC_3', 'VROC_10',
       'Vol_Spike_40', 'CMF_20', 'VIX', 'VROC_5', 'VIX_1_change',
       'Vol_Spike_10', 'Vol_Ratio_25', 'num_days_50', 'num_days_200',
       'OBV_Z5', 'Vol_Ratio_50', 'Max_120_Rows_Since', 'CMF_10', 'High',
       'SMA_100', 'ADL', 'VIX_10_change', 'Min_30_Rows_Since',
       'Close_Rel_Min200', 'Min_240_Rows_Since', 'SMA_200', 'Close',
       'max_min_first-30m', 'early_post_market_oc_pos_max',
       'late_pre_market_%_up_minutes', 'overnight_oc_neg_avg',
       'early_close_oc_neg_avg', 'midday_%_none_minutes',
       'post_market_other_oc_neg_avg', 'late_open_oc_pos_avg',
       'early_pre_market_oc_pos_max', 'midday_%_up_minutes',
       'overnight_oc_pos_avg', 'post_market_other_oc_pos_avg'],
      dtype=object)